In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import spark_partition_id
STORAGE_ACCOUNT = "storageaccount12344325"
container = "taxidata"
spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    dbutils.secrets.get(scope="snowflake", key="storage-account-key")
)
base       = f"abfss://{container}@{STORAGE_ACCOUNT}.dfs.core.windows.net"

raw_yellow = f"{base}/yellow/"
raw_lookup = f"{base}/zone_lookup/"
processed  = f"{base}/processed"


In [0]:
# (all 12 months)
df = spark.read.parquet(raw_yellow)
print("Row count ", df.count())


In [0]:
df.select(spark_partition_id()).distinct().count()


In [0]:
df.printSchema()


In [0]:
filtered_df = df.filter((F.col("tpep_pickup_datetime") < "2025-01-01") | (F.col("tpep_pickup_datetime") >= "2026-01-01"))
filtered_df.count()

In [0]:
# filtered_df.select("tpep_pickup_datetime", "tpep_dropoff_datetime").orderBy("tpep_pickup_datetime").show()

In [0]:
df_clean = df.filter(
    (F.col("tpep_pickup_datetime") >= "2025-01-01") & (F.col("tpep_pickup_datetime") < "2026-01-01"))

In [0]:
df_clean = df_clean.withColumn("pickup_date", F.to_date(F.col("tpep_pickup_datetime")))
print("Rows after cleaning:", df_clean.count())

In [0]:
df_clean.write.format("delta").mode("overwrite").partitionBy("pickup_date").save(f"{processed}/yellow_by_date")

In [0]:
spark.sql(f"DESCRIBE DETAIL delta.`{processed}/yellow_by_date`") \
     .select("numFiles", "sizeInBytes").show()

In [0]:
df = spark.read.format("delta").load(f"{processed}/yellow_by_date")
df = df.filter(
    (F.col("pickup_date") >= "2025-01-01") & (F.col("pickup_date") <= "2025-01-07"))

df.explain()
print("Rows in that week:", df.count())


In [0]:
# df.createOrReplaceTempView("taxi_data")

In [0]:
# %sql
# SELECT COUNT(*) AS total_count
# FROM taxi_data
# WHERE pickup_date >= '2025-01-01'
#   AND pickup_date <= '2025-01-07';